In [7]:
pip install pandas numpy faiss-cpu torch transformers openpyxl scikit-learn

In [16]:
import pandas as pd
import numpy as np
import faiss
import torch
import os
import re
import json # <-- Добавили импорт JSON

# --- КОНФИГУРАЦИЯ ---
EXCEL_FILE_PATH = "LK_modified.xlsx"
SHEET_NAME = 'Вопрос ответ'
QUESTION_COLUMN = 'Вопрос'
ANSWER_COLUMN = 'Ответ'
MODEL_NAME = "Tochka-AI/ruRoPEBert-e5-base-2k"
FAISS_INDEX_PATH = "qa_index.faiss"
ANSWERS_PATH = "answers.json"  # <-- Изменили расширение файла
SIMILARITY_THRESHOLD = 0.4

# --- Глобальные переменные ---
tokenizer = None
model = None
faiss_index = None
answers = None # Теперь это будет Python list, а не NumPy array

# --- Функции ---

# get_embeddings остается без изменений

def build_and_save_index():
    """
    Загружает данные из Excel, генерирует эмбеддинги, создает индекс FAISS,
    сохраняет индекс и соответствующие ответы в JSON.
    """
    global answers # Добавим сюда, чтобы сразу обновить глоб. переменную, если нужно
    print(f"Loading data from '{EXCEL_FILE_PATH}', sheet '{SHEET_NAME}'...")
    try:
        df = pd.read_excel(EXCEL_FILE_PATH, sheet_name=SHEET_NAME)
    except FileNotFoundError:
        print(f"Error: File not found at {EXCEL_FILE_PATH}")
        return False # Возвращаем признак неудачи
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return False

    if QUESTION_COLUMN not in df.columns or ANSWER_COLUMN not in df.columns:
        print(f"Error: Required columns ('{QUESTION_COLUMN}', '{ANSWER_COLUMN}') not found in sheet '{SHEET_NAME}'.")
        print(f"Available columns: {df.columns.tolist()}")
        return False

    df.dropna(subset=[QUESTION_COLUMN, ANSWER_COLUMN], inplace=True)
    df = df[df[QUESTION_COLUMN].astype(str).str.strip() != '']

    questions = df[QUESTION_COLUMN].astype(str).tolist()
    current_answers = df[ANSWER_COLUMN].tolist() # Это уже Python list

    if not questions:
        print("Error: No valid questions found after filtering.")
        return False

    embeddings = get_embeddings(questions)
    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    print(f"Saving FAISS index to '{FAISS_INDEX_PATH}'...")
    faiss.write_index(index, FAISS_INDEX_PATH)

    # --- ИЗМЕНЕНИЕ ЗДЕСЬ: Сохраняем ответы в JSON ---
    print(f"Saving answers list to '{ANSWERS_PATH}'...")
    try:
        with open(ANSWERS_PATH, 'w', encoding='utf-8') as f:
            json.dump(current_answers, f, ensure_ascii=False, indent=4) # ensure_ascii=False для кириллицы
        answers = current_answers # Обновляем глоб. переменную сразу
        print("Index and answers saved successfully.")
        return True # Возвращаем признак успеха
    except Exception as e:
        print(f"Error saving answers to JSON file: {e}")
        return False
    # --- КОНЕЦ ИЗМЕНЕНИЯ ---


def find_answer(user_query):
    """
    Принимает запрос пользователя, находит наиболее похожий вопрос в индексе FAISS
    и возвращает соответствующий ответ (загружая ответы из JSON), если сходство выше порога.
    """
    global faiss_index, answers, tokenizer, model

    # --- Загрузка ресурсов при первом вызове ---
    # Загрузка индекса FAISS (без изменений)
    if faiss_index is None:
        if not os.path.exists(FAISS_INDEX_PATH):
            print(f"Error: FAISS index file '{FAISS_INDEX_PATH}' not found. Attempting to build index...")
            if not build_and_save_index(): # Пытаемся построить индекс
                 return "Критическая ошибка: Не удалось создать или найти базу знаний."
            # Если build_and_save_index() был успешен, faiss_index может быть еще не загружен,
            # но файлы созданы. Попробуем загрузить снова.
        try:
            print(f"Loading FAISS index from '{FAISS_INDEX_PATH}'...")
            faiss_index = faiss.read_index(FAISS_INDEX_PATH)
            print("Index loaded.")
        except Exception as e:
             print(f"Error loading FAISS index even after trying to build it: {e}")
             return "Критическая ошибка: Не удалось загрузить базу знаний."


    # --- ИЗМЕНЕНИЕ ЗДЕСЬ: Загрузка ответов из JSON ---
    if answers is None:
        if not os.path.exists(ANSWERS_PATH):
             # Если индекса не было, build_and_save_index уже был вызван и должен был создать файл
             # Если файл все еще не существует, значит build_and_save_index не отработал
              print(f"Error: Answers file '{ANSWERS_PATH}' not found. Index building might have failed earlier.")
              return "Ошибка: База ответов не найдена. Возможно, не удалось создать индекс."
        try:
            print(f"Loading answers from '{ANSWERS_PATH}'...")
            with open(ANSWERS_PATH, 'r', encoding='utf-8') as f:
                answers = json.load(f) # Загружаем из JSON
            print("Answers loaded.")
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON from answers file '{ANSWERS_PATH}': {e}")
            return "Ошибка: Не удалось прочитать файл ответов. Файл может быть поврежден."
        except Exception as e:
            print(f"Error loading answers file '{ANSWERS_PATH}': {e}")
            return "Ошибка: Не удалось загрузить файл ответов."
    # --- КОНЕЦ ИЗМЕНЕНИЯ ---

    # Загрузка модели (без изменений)
    if tokenizer is None or model is None:
        print("Loading embedding model for query...")
        # Добавим try-except и для загрузки модели на всякий случай
        try:
            tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
            model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
            model.eval()
            print("Model loaded.")
        except Exception as e:
            print(f"CRITICAL ERROR: Could not load the embedding model '{MODEL_NAME}'. Error: {e}")
            return "Критическая ошибка: Невозможно загрузить модель для обработки запроса."


    cleaned_query = user_query # Используем как есть, без clean_text

    # Получение эмбеддинга запроса и поиск (без изменений)
    try:
        query_embedding = get_embeddings([cleaned_query])
        faiss.normalize_L2(query_embedding)

        k = 1
        distances, indices = faiss_index.search(query_embedding, k)

        best_match_index = indices[0][0]
        similarity_score = distances[0][0]

        print(f"Query: '{user_query}'")
        print(f"Best match index: {best_match_index}, Similarity score: {similarity_score:.4f}")

        if similarity_score >= SIMILARITY_THRESHOLD:
            # Индексация работает так же для списка, как и для NumPy array
            found_answer = answers[best_match_index]
            return found_answer
        else:
            return "Извините, я не смог найти точный ответ на ваш вопрос в базе знаний."
    except Exception as e:
        # Общий обработчик ошибок на случай проблем при эмбеддинге или поиске
        print(f"Error during query embedding or search: {e}")
        return "Произошла ошибка при обработке вашего запроса."


# --- Пример использования (добавим проверку результата build_and_save_index) ---
if __name__ == "__main__":

    index_ready = False
    if os.path.exists(FAISS_INDEX_PATH) and os.path.exists(ANSWERS_PATH):
        print("Index and answers files already exist.")
        index_ready = True
    else:
        print("Index or answers file not found. Building index...")
        if build_and_save_index():
            index_ready = True
        else:
            print("Failed to build index. Cannot proceed with search.")

    if index_ready:
        print("\n--- Testing search function ---")

        test_query_1 = "Как добавить автомобиль в систему?"
        answer_1 = find_answer(test_query_1)
        print(f"Answer 1: {answer_1}\n")

        test_query_2 = "Когда сука"
        answer_2 = find_answer(test_query_2)
        print(f"Answer 2: {answer_2}\n")

        test_query_3 = "какая сегодня погода?"
        answer_3 = find_answer(test_query_3)
        print(f"Answer 3: {answer_3}\n")
    else:
         print("\nSkipping search tests because index is not ready.")

Index and answers files already exist.

--- Testing search function ---
Loading FAISS index from 'qa_index.faiss'...
Index loaded.
Loading answers from 'answers.json'...
Answers loaded.
Loading embedding model for query...
Model loaded.
Generating embeddings for 1 texts...
Processed batch 1 / 1
Embeddings generated.
Query: 'Как добавить автомобиль в систему?'
Best match index: 4, Similarity score: 0.7683
Answer 1: Для внесения данных по личному автомобилю обратитесь, пожалуйста, к своему руководителю для создания заявки по теме "Изменение режима характера работы", подтема "Установка РХР и топливной карты". В комментариях опишите ситуацию и приложите ПТС, СТС, страховой полис и водительское удостоверение.

Generating embeddings for 1 texts...
Processed batch 1 / 1
Embeddings generated.
Query: 'Когда сука зарплата'
Best match index: 138, Similarity score: 0.6527
Answer 2: Заработная плата в нашей компании выплачивается двумя частями. 28-го числа месяца направляется аванс, 13-го числа мес